# Supermarket Sales Analytics with AI

**Student:** Mohammed Hilal  
**Degree:** B.E. Computer Science & Engineering (AI & ML)  
**Institution:** Nawab Shah Alam Khan College of Engineering & Technology, Hyderabad  
**Internship:** AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026  
**Project type:** Exploratory Data Analysis, Business Insights and Customer Churn Prediction

## Project objective
Analyze supermarket transaction data, clean and validate the data, identify sales and customer patterns, generate business insights, and build a simple leakage-aware churn prediction model using RFM-style customer features.

The notebook is designed to run even when the original internship dataset is unavailable: if `supermarket_sales.csv` is not present, it creates a reproducible 500-row demonstration dataset with the same type of fields used in the Masterclass 4 workflow.


## Masterclass alignment
- **Masterclass 1:** raw data → cleaning → business questions
- **Masterclass 2:** EDA → visualizations → observations → insights → recommendations
- **Masterclass 3:** RFM features → churn definition → leakage check → Logistic Regression → evaluation
- **Masterclass 4:** AI-assisted project structure → analytics → reusable outputs


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')


## 1. Load the dataset
The notebook first looks for `supermarket_sales.csv`. If it is not available, a reproducible demonstration dataset is generated. This makes the submission self-contained while still allowing the internship dataset to be substituted later.


In [ ]:
DATA_PATH = Path('supermarket_sales.csv')

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print('Loaded external dataset:', DATA_PATH)
else:
    rng = np.random.default_rng(RANDOM_STATE)
    n = 500
    products = {
        'Phone': ('Electronics', 22000), 'Laptop': ('Electronics', 55000),
        'Headphones': ('Electronics', 3500), 'TV': ('Electronics', 42000),
        'Mixer': ('Home Appliances', 4500), 'Vacuum Cleaner': ('Home Appliances', 7000),
        'Shirt': ('Fashion', 1800), 'Shoes': ('Fashion', 3200),
        'Makeup Kit': ('Beauty', 2500), 'Skincare': ('Beauty', 1800)
    }
    product_names = list(products)
    customers = [f'C{idx:03d}' for idx in range(1, 101)]
    branches = ['A', 'B', 'C']
    regions = {'A': 'Hyderabad', 'B': 'Bengaluru', 'C': 'Pune'}
    dates = pd.date_range('2026-01-01', '2026-12-31', freq='D')
    rows = []
    for i in range(n):
        customer = rng.choice(customers)
        product = rng.choice(product_names)
        category, base_price = products[product]
        branch = rng.choice(branches, p=[0.32, 0.30, 0.38])
        quantity = int(rng.integers(1, 6))
        unit_price = round(base_price * rng.uniform(0.85, 1.15), 2)
        discount = round(rng.uniform(0, 0.15), 3)
        total_sales = round(quantity * unit_price * (1 - discount), 2)
        rows.append([
            f'INV{i+1:04d}', rng.choice(dates), customer, branch, regions[branch],
            rng.choice(['New', 'Returning'], p=[0.25, 0.75]),
            rng.choice(['Male', 'Female']), product, category, quantity,
            unit_price, discount, rng.choice(['UPI', 'Credit Card', 'Cash', 'Debit Card']),
            round(rng.uniform(3.0, 5.0), 1), total_sales
        ])
    df = pd.DataFrame(rows, columns=[
        'Invoice_ID','Date','Customer_Type_ID','Branch','Region','Customer_Type',
        'Gender','Product','Category','Quantity','Unit_Price','Discount',
        'Payment_Method','Rating','Total_Sales'
    ])
    # Add a small amount of realistic missingness for the cleaning stage.
    missing_idx = rng.choice(df.index, size=8, replace=False)
    df.loc[missing_idx[:4], 'Rating'] = np.nan
    df.loc[missing_idx[4:], 'Region'] = np.nan

print('Shape:', df.shape)
df.head()


## 2. Data understanding and cleaning
We inspect data types, missing values and duplicate records before analysis.


In [ ]:
print(df.info())
print('\nMissing values:')
display(df.isna().sum().sort_values(ascending=False).to_frame('missing'))
print('\nDuplicate rows:', df.duplicated().sum())


In [ ]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
for col in ['Quantity', 'Unit_Price', 'Discount', 'Rating', 'Total_Sales']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['Region'] = df['Region'].fillna('Unknown')
df['Rating'] = df['Rating'].fillna(df['Rating'].median())
df = df.drop_duplicates().copy()

print('Cleaned shape:', df.shape)
print('Remaining missing values:', int(df.isna().sum().sum()))


## 3. Business questions
1. Which products and categories generate the most sales?
2. Which branch/region performs best?
3. What are the monthly sales trends?
4. How do customer types differ in sales contribution?
5. Which customers are high-value or at higher churn risk?
6. What actions can management consider based on the analysis?


In [ ]:
kpis = pd.Series({
    'Total Sales': df['Total_Sales'].sum(),
    'Average Order Value': df['Total_Sales'].mean(),
    'Transactions': len(df),
    'Unique Customers': df['Customer_Type_ID'].nunique(),
    'Average Rating': df['Rating'].mean()
})
display(kpis.to_frame('Value'))


In [ ]:
category_sales = df.groupby('Category')['Total_Sales'].sum().sort_values(ascending=False)
product_sales = df.groupby('Product')['Total_Sales'].sum().sort_values(ascending=False)
branch_sales = df.groupby('Branch')['Total_Sales'].sum().sort_values(ascending=False)
region_sales = df.groupby('Region')['Total_Sales'].sum().sort_values(ascending=False)
customer_type_sales = df.groupby('Customer_Type')['Total_Sales'].agg(['sum','mean','count']).sort_values('sum', ascending=False)

display(category_sales.to_frame('Total Sales'))
display(product_sales.head(10).to_frame('Total Sales'))
display(branch_sales.to_frame('Total Sales'))
display(customer_type_sales)


## 4. Exploratory visualizations


In [ ]:
monthly_sales = df.set_index('Date').resample('ME')['Total_Sales'].sum()
plt.figure(figsize=(10,5)); monthly_sales.plot(marker='o'); plt.title('Monthly Sales Trend'); plt.xlabel('Month'); plt.ylabel('Sales'); plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(9,5)); category_sales.plot(kind='bar'); plt.title('Sales by Category'); plt.xlabel('Category'); plt.ylabel('Sales'); plt.xticks(rotation=35); plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(9,5)); product_sales.head(10).sort_values().plot(kind='barh'); plt.title('Top 10 Products by Sales'); plt.xlabel('Sales'); plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(7,5)); branch_sales.plot(kind='bar'); plt.title('Sales by Branch'); plt.xlabel('Branch'); plt.ylabel('Sales'); plt.tight_layout(); plt.show()


In [ ]:
customer_type_sales['sum'].plot(kind='pie', autopct='%1.1f%%', figsize=(6,6)); plt.title('Sales Contribution by Customer Type'); plt.ylabel(''); plt.show()


## 5. Observations, insights and recommendations


In [ ]:
observations = {
    '1': f"Top category by sales: {category_sales.index[0]}.",
    '2': f"Top product by sales: {product_sales.index[0]}.",
    '3': f"Top branch by sales: {branch_sales.index[0]}.",
    '4': f"Top customer type by sales: {customer_type_sales.index[0]}.",
    '5': f"Highest-sales month: {monthly_sales.idxmax().strftime('%B %Y')}."
}
for k,v in observations.items(): print(f"Observation {k}: {v}")

print('\nInsights:')
print('- Category/product concentration can guide inventory and promotional planning.')
print('- Branch-level differences can help management compare local performance.')
print('- Monthly variation can support staffing, inventory and cash-flow planning.')
print('- Customer-type contribution can inform retention and acquisition strategies.')

print('\nRecommendations:')
print('- Monitor high-performing products and categories for stock availability.')
print('- Investigate underperforming branches and compare their product mix.')
print('- Plan inventory and staffing around high-sales periods.')
print('- Develop targeted retention offers for valuable repeat customers while continuing new-customer acquisition.')


## 6. Customer-level RFM analysis
RFM features convert transaction-level data into customer-level information: Recency, Frequency and Monetary value.


In [ ]:
snapshot_date = df['Date'].max() + pd.Timedelta(days=1)
rfm = df.groupby('Customer_Type_ID').agg(
    Recency=('Date', lambda x: (snapshot_date - x.max()).days),
    Frequency=('Invoice_ID', 'nunique'),
    Monetary=('Total_Sales', 'sum'),
    Average_Order_Value=('Total_Sales', 'mean')
).reset_index()
rfm.head()


## 7. Leakage-aware churn prediction
For a real production project, churn should be defined using a future observation window. This notebook uses a transparent demonstration label based on recency and explicitly excludes Recency from the model features to avoid directly feeding the rule that created the target into the model.


In [ ]:
CHURN_DAYS = 180
rfm['Churn'] = (rfm['Recency'] >= CHURN_DAYS).astype(int)
print('Churn distribution:')
display(rfm['Churn'].value_counts().rename(index={0:'Not Churn',1:'Churn'}).to_frame('Customers'))

features = ['Frequency', 'Monetary', 'Average_Order_Value']
X = rfm[features]
y = rfm['Churn']

if y.nunique() < 2 or y.value_counts().min() < 2:
    print('Not enough class variation for a stable demonstration model. Adjust CHURN_DAYS or use the internship dataset.')
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('logreg', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE))
    ])
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print('Accuracy :', round(accuracy_score(y_test, y_pred), 3))
    print('Precision:', round(precision_score(y_test, y_pred, zero_division=0), 3))
    print('Recall   :', round(recall_score(y_test, y_pred, zero_division=0), 3))
    print('F1       :', round(f1_score(y_test, y_pred, zero_division=0), 3))
    print('\nClassification report:')
    print(classification_report(y_test, y_pred, zero_division=0))
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5,4)); sns.heatmap(cm, annot=True, fmt='d', cmap='Blues'); plt.title('Confusion Matrix'); plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.tight_layout(); plt.show()


## 8. Final business interpretation
- Predictive outputs should be treated as risk signals, not guarantees.
- Model evaluation should consider precision and recall, not accuracy alone.
- The churn label used here is a demonstration rule; a production project should use a clearly defined future observation window.
- Business recommendations should be supported by the observed data and model evidence.

## Conclusion
The project demonstrates a complete analytics workflow from data preparation and EDA to customer-level feature engineering and a basic churn classification model. It also follows the internship's AI-assisted development philosophy: AI can accelerate coding and analysis, but the analyst remains responsible for validation, interpretation and final decisions.
